# 03 – Content-Based Music Recommender
**Project**: Spotify Music Recommender & Analysis  
**Author**: S33mi  
**Goal**: Build a practical content-based recommender using the scaled audio features prepared in notebook 02.

We implement two approaches:
1. **Cosine Similarity** (classic, good for understanding)
2. **Nearest Neighbors (KNN)** – more scalable and recommended for the full dataset

## 1. Setup & Imports

In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors
import joblib
import warnings

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.max_columns", 30)

print("Libraries loaded successfully.")

Libraries loaded successfully.


## 2. Load Processed Data from Notebook 02

In [13]:
# Load the combined dataframe (metadata + scaled features)
final_df = pd.read_csv("data/processed/spotify_features_scaled.csv")

# Load pure feature matrix
X = np.load("data/processed/X_scaled.npy")

# Load feature column names and scaler (useful later)
feature_cols = joblib.load("models/feature_columns.pkl")
scaler = joblib.load("models/standard_scaler.pkl")

print(f"Data shape          : {final_df.shape}")
print(f"Feature matrix shape: {X.shape}")
print(f"Feature columns     : {feature_cols}")

# Quick look
final_df[["track_name", "artists", "track_genre", "popularity"] + feature_cols].head()

Data shape          : (89741, 16)
Feature matrix shape: (89741, 10)
Feature columns     : ['danceability', 'energy', 'loudness', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_min']


,track_name,artists,track_genre,popularity,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_min
0,Comedy,Gen Hoshino,acoustic,73,0.644260,-0.675976,0.335731,0.490464,-0.875177,-0.535478,0.723666,0.934036,-1.133609,0.013495
1,Ghost - Acoustic,Ben Woodward,acoustic,55,-0.804604,-1.825609,-1.673094,-0.098361,1.760797,-0.535464,-0.595072,-0.770280,-1.479854,-0.704151
2,To Begin Again,Ingrid Michaelson;ZAYN,acoustic,57,-0.702731,-1.073476,-0.236523,-0.280217,-0.349638,-0.535481,-0.512971,-1.329508,-1.518271,-0.162163
3,Can't Help Falling In Love,Kina Grannis,acoustic,71,-1.676186,-2.240257,-1.918236,-0.451480,1.704637,-0.535263,-0.436002,-1.242010,1.981637,-0.240899
4,Hold On,Chord Overstreet,acoustic,82,0.316001,-0.746123,-0.226373,-0.307584,0.415912,-0.535481,-0.687948,-1.150708,-0.070037,-0.268168


## 3. Helper: Search for a Track
Useful because many tracks can share similar names.

In [14]:
def search_track(query: str, top_n: int = 10):
    """
    Simple case-insensitive search on track_name and artists.
    Returns a small dataframe of matching tracks.
    """
    query = query.lower()
    mask = (
        final_df["track_name"].str.lower().str.contains(query, na=False) |
        final_df["artists"].str.lower().str.contains(query, na=False)
    )
    results = final_df.loc[mask, ["track_id", "track_name", "artists", "track_genre", "popularity"]]
    return results.head(top_n)

# Example
print("Search example – 'blinding':")
display(search_track("blinding"))

Search example – 'blinding':


,track_id,track_name,artists,track_genre,popularity
13198,6VXMal8tPWfZl0KOIwp359,Blinding Lights,Kidz Bop Kids,children,0
22124,7JAo7wy8BzmP9smtTJ3fuU,Blinding Lights,Lucas Estrada;Twan Ray,deep-house,51
28043,0wI0S42Cg41DVGqIIVimTM,Blinding Lights,Revelries;Victoria Voss,edm,64
40692,2pUI7cFLOYqtNQska1JEEI,Blinding Lights (Instrumental Guitar),Guus Dielissen,guitar,43
67618,0VjIjW4GlUZAMYd2vXMi3b,Blinding Lights,The Weeknd,pop,91
67672,7LltNXuqCBGOAp1iwmAmB3,Blinding Lights,The Weeknd,pop,0
67673,5Dt9HFzeBVmFtrk3BTiDEn,Blinding Lights,The Weeknd,pop,3
67674,4O0ymDK32zylHELT506JPI,Blinding Lights,The Weeknd,pop,0
67676,6h49LElLEJLbKfR6zdcFtf,Blinding Lights,The Weeknd,pop,3
70324,56GH1HDx3NzS4wDDdNQVe9,Blinding Lights,All Time Low,punk-rock,1


## 4. Approach A – Cosine Similarity (Full Matrix)

**Note**: Building a full N×N similarity matrix for ~90–100k tracks is memory-heavy.  
We demonstrate it on a **sample** first, then move to the scalable KNN version.

In [15]:
# Take a manageable sample for demonstration
sample_size = 8000
sample_idx = np.random.choice(len(final_df), size=sample_size, replace=False)
sample_df = final_df.iloc[sample_idx].reset_index(drop=True)
X_sample = X[sample_idx]

print(f"Working with sample of {sample_size} tracks for full cosine matrix demo.")

# Compute cosine similarity matrix
cosine_sim = cosine_similarity(X_sample)
print(f"Cosine similarity matrix shape: {cosine_sim.shape}")

Working with sample of 8000 tracks for full cosine matrix demo.
Cosine similarity matrix shape: (8000, 8000)


In [16]:
def recommend_cosine(track_name: str, top_n: int = 10, sample_df=sample_df, sim_matrix=cosine_sim):
    """
    Recommend songs using pre-computed cosine similarity (sample version).
    """
    # Find the track (take first match)
    matches = sample_df[sample_df["track_name"].str.lower() == track_name.lower()]
    if matches.empty:
        # fallback to partial match
        matches = sample_df[sample_df["track_name"].str.lower().str.contains(track_name.lower(), na=False)]

    if matches.empty:
        print(f"Track '{track_name}' not found in the sample.")
        return None

    idx = matches.index[0]
    sim_scores = list(enumerate(sim_matrix[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Exclude the track itself
    sim_scores = sim_scores[1:top_n+1]
    track_indices = [i[0] for i in sim_scores]

    recommendations = sample_df.iloc[track_indices][
        ["track_name", "artists", "track_genre", "popularity"]
    ].copy()
    recommendations["similarity"] = [i[1] for i in sim_scores]

    return recommendations.reset_index(drop=True)

### Test Cosine Recommender (on sample)

In [19]:
# Pick a track that exists in the sample (you can change this)
# Sample -> "Comedy", "Sevmekten Kim Usanır", "Blinding Lights", "Atrakce", "Rain", "Lo-fi", "The Steeple","The Resistance"
example_track = sample_df.iloc[42]["track_name"]
print(f"Seed track: {example_track}")
print(f"Artist    : {sample_df.iloc[42]['artists']}")
print(f"Genre     : {sample_df.iloc[42]['track_genre']}\n")

recs = recommend_cosine(example_track, top_n=8)
display(recs)

Seed track: 離別的規矩
Artist    : Jer 柳應廷
Genre     : cantopop



,track_name,artists,track_genre,popularity,similarity
0,Sevmekten Kim Usanır,Müzeyyen Senar,j-rock,39,0.969085
1,Alaturka,Sezen Aksu,turkish,39,0.968868
2,Have Yourself A Merry Little Christmas,The Jackson 5,soul,0,0.945332
3,Yağmur (Denizhan'la Düet),Cem Adrian,turkish,37,0.932744
4,Reflexo (feat. Eliana Ribeiro),Toca de Assis;Eliana Ribeiro,brazil,43,0.925751
5,Calma (Todo Dia Mal Passa),Lorhann,brazil,44,0.917734
6,Let It Be - Remastered 2015,The Beatles,british,63,0.917064
7,如果雨之後,Eric Chou,mandopop,61,0.915753


## 5. Approach B – Nearest Neighbors (Recommended – Scalable)

In [20]:
# Fit NearestNeighbors on the FULL scaled feature matrix
# Using cosine distance (equivalent to 1 - cosine similarity)
knn = NearestNeighbors(
    n_neighbors=15,          # we will request top 10 + some buffer
    metric="cosine",
    algorithm="brute",       # exact, works well for this size
    n_jobs=-1
)

knn.fit(X)
print("NearestNeighbors model fitted on full dataset.")

NearestNeighbors model fitted on full dataset.


In [21]:
def recommend_knn(
    track_name: str = None,
    track_id: str = None,
    top_n: int = 10,
    return_distance: bool = True
):
    """
    Recommend similar tracks using KNN (cosine distance).

    Provide either track_name or track_id.
    """
    if track_id is not None:
        matches = final_df[final_df["track_id"] == track_id]
    elif track_name is not None:
        matches = final_df[final_df["track_name"].str.lower() == track_name.lower()]
        if matches.empty:
            matches = final_df[final_df["track_name"].str.lower().str.contains(track_name.lower(), na=False)]
    else:
        raise ValueError("Please provide either track_name or track_id.")

    if matches.empty:
        print("Track not found.")
        return None

    # Take the first match
    seed = matches.iloc[0]
    seed_idx = matches.index[0]

    print(f"Seed Track : {seed['track_name']}")
    print(f"Artists    : {seed['artists']}")
    print(f"Genre      : {seed['track_genre']}")
    print(f"Popularity : {seed['popularity']}\n")

    # Get neighbors
    distances, indices = knn.kneighbors(
        X[seed_idx].reshape(1, -1),
        n_neighbors=top_n + 1
    )

    # Remove the seed itself (distance ~0)
    neighbor_indices = indices[0][1:]
    neighbor_distances = distances[0][1:]

    recs = final_df.iloc[neighbor_indices][
        ["track_id", "track_name", "artists", "track_genre", "popularity"]
    ].copy()

    if return_distance:
        # Convert cosine distance to similarity for easier reading
        recs["cosine_similarity"] = 1 - neighbor_distances

    return recs.reset_index(drop=True)

### Test KNN Recommender

In [22]:
# Example 1 – search first, then recommend
search_results = search_track("blinding lights", top_n=5)
display(search_results)

,track_id,track_name,artists,track_genre,popularity
13198,6VXMal8tPWfZl0KOIwp359,Blinding Lights,Kidz Bop Kids,children,0
22124,7JAo7wy8BzmP9smtTJ3fuU,Blinding Lights,Lucas Estrada;Twan Ray,deep-house,51
28043,0wI0S42Cg41DVGqIIVimTM,Blinding Lights,Revelries;Victoria Voss,edm,64
40692,2pUI7cFLOYqtNQska1JEEI,Blinding Lights (Instrumental Guitar),Guus Dielissen,guitar,43
67618,0VjIjW4GlUZAMYd2vXMi3b,Blinding Lights,The Weeknd,pop,91


In [23]:
# Use one of the track_ids or names from the search
if not search_results.empty:
    example_id = search_results.iloc[0]["track_id"]
    recs_knn = recommend_knn(track_id=example_id, top_n=10)
    display(recs_knn)
else:
    # Fallback – pick any popular track
    popular = final_df.nlargest(1, "popularity")
    recs_knn = recommend_knn(track_id=popular.iloc[0]["track_id"], top_n=10)
    display(recs_knn)

Seed Track : Blinding Lights
Artists    : Kidz Bop Kids
Genre      : children
Popularity : 0



,track_id,track_name,artists,track_genre,popularity,cosine_similarity
0,6wtELTbJEAkbugVDSKa0sh,Atrakce,PAWLIE POIZN;Medooza,emo,36,0.989883
1,1MOOJuxUu9QiQE9GgkYYPb,My Person,Spencer Crandall,country,61,0.987022
2,1sQie71FoBxfYusuTDOXAA,Mob Rule,Bad//Dreems,garage,29,0.986477
3,1K61P0kbiT6cJzh77NnxFg,My Person,Spencer Crandall,country,55,0.986402
4,0SFETrpna5wpB1ibBPPyWI,Me Voy,Los Victorios,ska,34,0.985518
5,3r6AJfqJ44FepL26lwLMPf,1+1,Ready Kirken,folk,46,0.980496
6,2g9UPl5sdCGAcszvwMhE5o,Rock de la Cárcel,Palito Ortega,rock-n-roll,33,0.979025
7,5RdZcipSf2Xysg02sVsoEp,La Alegría en Serio,Los Caligaris,ska,40,0.977845
8,4cNOOkWBjY41prOyd2r6Jz,Manohari,Divya Kumar;Neeti Mohan,folk,55,0.977239
9,4haXv9IJwOlJJsT6FNJPYk,The Middle,Jimmy Eat World,alt-rock,0,0.976680


In [24]:
# Another example – by name
recs2 = recommend_knn(track_name="Shape of You", top_n=8)
display(recs2)

Seed Track : Shape Of You
Artists    : Andrew Foy
Genre      : guitar
Popularity : 24



,track_id,track_name,artists,track_genre,popularity,cosine_similarity
0,0o30XkI520zpdZIeSRaSXr,"Overture in the French Style, Op. 2, BWV 831 (Excerpts): XI. Echo",Johann Sebastian Bach;Orion Weiss,classical,0,0.970357
1,6dDnNol7rIKv6KO6JzV8jk,Old Pine Box,Andrew Marlin,bluegrass,21,0.965143
2,7snsIXBqXSt3XcYAJUiBNr,This Here,Bobby Timmons,piano,32,0.964491
3,5QFoUJ4XD88IEA5FYst3e6,Amaneci En Tus Brazos,Chamin Madero,guitar,20,0.962882
4,6IWumNs0oyQfa8mLd5rjcn,Voila!,Stuart Duncan;Edgar Meyer;Chris Thile;Yo-Yo Ma,bluegrass,23,0.962263
5,7jmKrl6M28CmP0eUEd2J2k,Minha Saudade,João Gilberto,guitar,23,0.961582
6,4LGOSAiawQeYRefjruL9Xe,Brasileirinho,Eudóxia De Barros,piano,27,0.958165
7,53yCbC8zjcNQi5534UDoWL,Memories (Instrumental Guitar),Guus Dielissen,acoustic,37,0.958030


## 6. Recommend from a Custom Feature Vector
Useful later if you want to feed a new song’s audio features.

In [25]:
def recommend_from_features(feature_vector, top_n: int = 10):
    """
    feature_vector: 1D array-like of length = len(feature_cols)
                    (already scaled with the same StandardScaler)
    """
    feature_vector = np.array(feature_vector).reshape(1, -1)

    distances, indices = knn.kneighbors(feature_vector, n_neighbors=top_n)

    recs = final_df.iloc[indices[0]][
        ["track_name", "artists", "track_genre", "popularity"]
    ].copy()
    recs["cosine_similarity"] = 1 - distances[0]

    return recs.reset_index(drop=True)

# Example: take an existing track’s features and get neighbors (should return itself first)
example_features = X[100]
recs_custom = recommend_from_features(example_features, top_n=6)

display(recs_custom)

,track_name,artists,track_genre,popularity,cosine_similarity
0,Rain,Motohiro Hata,acoustic,58,1.000000
1,Woh Din (Film Version),Pritam;TUSHAR JOSHI,indian,46,0.965323
2,"Woh Din (From ""Chhichhore"")",Pritam;TUSHAR JOSHI,indian,50,0.965323
3,Manwa Laage,Shreya Ghoshal;Arijit Singh,pop-film,60,0.960565
4,A Veces Me Pregunto,DLG,salsa,29,0.956166
5,Jogi,Yasser Desai;Aakanksha Sharma,pop-film,67,0.955675


## 7. Save the Recommender Artifacts

In [26]:
import os
os.makedirs("models", exist_ok=True)

# Save the fitted KNN model
joblib.dump(knn, "models/knn_recommender.pkl")

# Also save a lightweight version of the metadata for easy lookup
meta_for_rec = final_df[["track_id", "track_name", "artists", "track_genre", "popularity"]].copy()
meta_for_rec.to_csv("data/processed/meta_for_recommender.csv", index=False)

print("Saved:")
print("  - models/knn_recommender.pkl")
print("  - data/processed/meta_for_recommender.csv")

Saved:
  - models/knn_recommender.pkl
  - data/processed/meta_for_recommender.csv


## 8. Quick Evaluation Ideas (Manual)

Content-based recommenders are usually evaluated qualitatively or with:
- Genre consistency of recommendations
- Diversity (how different the top-N are)
- User studies / listening tests

You can also compute average genre match rate for a sample of seeds.



In [27]:
def genre_match_rate(seed_genre, recommendations):
    if recommendations is None or recommendations.empty:
        return 0.0
    matches = (recommendations["track_genre"] == seed_genre).sum()
    return matches / len(recommendations)

# Example check
seed_genre = final_df.iloc[0]["track_genre"]
recs_check = recommend_knn(track_id=final_df.iloc[0]["track_id"], top_n=10)
print(f"Seed genre: {seed_genre}")
print(f"Genre match rate in top-10: {genre_match_rate(seed_genre, recs_check):.2%}")

Seed Track : Comedy
Artists    : Gen Hoshino
Genre      : acoustic
Popularity : 73

Seed genre: acoustic
Genre match rate in top-10: 20.00%


## Summary

- Built a **scalable KNN-based content recommender** using cosine distance on StandardScaled audio features.
- Provided helper functions for search + recommendation by name or track_id.
- Saved the model and metadata for use in a future demo (Gradio / Streamlit) or the playlist optimization project.

**Next notebook**: `04_clustering_mood.ipynb`  
→ Discover natural groups / mood playlists with K-Means (and optionally hierarchical clustering).